# Ⅰ第5回 宿題：転移学習を図解する（知識の整理）

**問い**：転移学習で，何を・なぜ・どこまで学習し直すのか．自分で調べて整理し，Google スライドで図解する．

- 授業のスライドは例の 1 つ．**授業スライド以外の資料を 2 つ以上**使って調べる
- 提出物：この `summary.ipynb` と，Google スライドを PDF にした `c1_d5_summary.pdf`
- 期限：11/5（木）13:10．「第5回 課題提出」へ

## (1) 調べる

資料の例（ほかの資料でもよい）：

- 教科書 第8章「転移学習プログラミング」
- PyTorch 公式チュートリアル「Transfer Learning for Computer Vision Tutorial」：https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html
- スタンフォード大学の講義ノート CS231n「Transfer Learning」：https://cs231n.github.io/transfer-learning/
- 論文 Yosinski ら（2014）「How transferable are features in deep neural networks?」：https://arxiv.org/abs/1411.1792

LLM（ChatGPT など）で調べてもよい．ただし LLM の答えは出典にならない．書いた内容は上のような資料で確かめ，その資料を出典に書く．

## (2) 整理する（スライドを作る前に書く）

このセルをダブルクリックして，＿＿ に書き込む．1 マス 1〜2 行．

| テーマ | 自分の言葉の説明 | 根拠 | スライド |
|---|---|---|---|
| 1. 転移学習とは | 大量の画像（ImageNet：約120万枚・1000種類）で学習済みのモデルを自分の少ない写真に使い回すこと。ゼロからは重みがでたらめな状態で始めるが、転移学習は良い特徴を持った状態から始められる。 | PyTorch 公式チュートリアル Transfer Learning for Computer Vision の冒頭 https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html | 1 枚目 |
| 2. 凍結・微調整・フルの違い | 凍結：最後の層だけ付け替えて学習し、ほかは固定。微調整：学習済みの重みを出発点に、全部の層を学習。呼び方は資料で違い、PyTorchは「fixed feature extractor」と「Finetuning」、CS231nは「fine-tuning」を一部の上の層だけでも全層でもよいとする。授業の微調整は全層なので「フルファインチューニング」にあたる。 | PyTorch 公式チュートリアル https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html ／CS231n「Transfer Learning」 https://cs231n.github.io/transfer-learning/ | 2 枚目 |
| 3. 役立つ理由 | 入口に近い層は線・色などどの画像にも共通する特徴を取り出し、出口に近い層は元のタスク専用の特徴になる。入口側は別のタスクでも使えるので、出口の層だけ付け替えれば足りる。 | 論文 Yosinski ら（2014）「How transferable are features in deep neural networks?」：https://arxiv.org/abs/1411.1792 | 3 枚目 |
| 4. 学習率を小さくする理由 | 学習済みの重みはすでに良い状態なので、大きな学習率で全部を動かすと壊れてしまう。CS231nでも微調整する層は新しい層より小さい学習率にするのが普通とされる。 | CS231n「Transfer Learning」 https://cs231n.github.io/transfer-learning/ | 4 枚目 |
| 5. 何層まで学習し直すか | データの量と元のデータとの似かたで決める。少なくて似ている→最後の層だけ（過学習しやすいので）、多くて似ている→全体を微調整、少なくて違う→入口寄りの層の特徴を使う。 | CS231n「Transfer Learning」 https://cs231n.github.io/transfer-learning/ ／論文 Yosinski ら（2014）「How transferable are features in deep neural networks?」：https://arxiv.org/abs/1411.1792 | 5 枚目 |
| 6. 自分の写真なら | 写真は200枚（訓練 140 枚）と少ない。ResNet18は凍結98.7％、微調整100.0％で差は1枚分未満なので凍結（最後の層だけ）を第一候補にする。 | (3) の表（ex2 の結果）／CS231n「Transfer Learning」 https://cs231n.github.io/transfer-learning/ | 6 枚目 |

## (3) 自分の写真ならどうするか

演習2（`ex2.ipynb`）の結果を写す．

| モデル | 平均 ± 標準偏差（%） | CNN との 差 ÷ 標準偏差 と判定 |
|---|---|---|
| CNN（ゼロから） | 84.0 ± 6.4 | ― |
| ResNet18 凍結 | 98.7 ± 1.8 | +14.7 ÷ 4.7 = 3.1 → 差がある |
| ResNet18 微調整 | 100.0 ± 0.0 | +16.0 ÷ 4.5 = 3.5 → 差がある |
| ViT 凍結 | 98.7 ± 1.8 | +14.7 ÷ 4.7 = 3.1 → 差がある |

- 自分の写真は，「写真が多い・少ない」×「ImageNet に似ている・似ていない」のどれに当たるか．理由：自分の写真は「写真が少ない」×「ImageNetに似ている」に当たると考える。訓練に使えるのは140枚と少ない。また、最後の層だけの凍結でも98.7％まで上がったので、ImageNetの特徴がそのまま使える程度に似ていると考えられる。
- ResNet18 なら，何層まで学習し直すのがよいと考えるか．理由：ResNet18は、最後の層（fc）だけ学習し直すのがよいと考える。微調整との差は1.3ポイントで、テスト1枚分（3.3 ポイント）より小さい。写真が少ないので、全部を学習すると過学習しやすい。
- それを確かめるには，どう実験するか（学習する層・回数・何で判定するか）：学習する層を3通り（fcだけ／layer4とfc／全部）にして、シード0〜4の5回ずつ回す。テスト正解率の平均と差 ÷ 標準偏差で判定する。テストが30枚と少ないので、枚数を増やすか検証正解率も合わせて見る。

## (4) Google スライドで図解する

1. Google スライドで **5〜7 枚** 作る．1 枚に 1 テーマ（(2) の 1〜6）と，最後に出典の 1 枚
2. 各スライドに **図を 1 つ以上** 入れる．自分で描いても，Google スライドの AI（Gemini）に描かせてもよい
3. AI に描かせた図は，層の数・矢印・文字の誤りを確かめて直す．直せない所は説明文で正しく補い，図の近くに「AI で生成」と書く
4. 説明文は自分の言葉で書く
5. 人の顔・個人が分かるものは載せない
6. 「ファイル → ダウンロード → PDF ドキュメント」で保存し，ファイル名を `c1_d5_summary.pdf` にする

## (5) 評価の観点（各 ○・△・×）

| 観点 | 見る所 |
|---|---|
| 1. 定義 | 用語の定義が正確で，資料による呼び方の違いに触れている |
| 2. 違い | ゼロから・凍結・微調整の違いを，出発点と学習する層で説明している |
| 3. 理由 | 学習率を小さくする理由・層の役割を，根拠つきで説明している |
| 4. 図 | 図に誤りが無い（AI で描いた図は誤りを直し，「AI で生成」と書いてある） |
| 5. 当てはめ | 自分の写真の結果（演習2）を使って，何層まで学習するかを考えている |
| 6. 出典 | 授業スライド以外の資料を 2 つ以上，名前と URL（または章）で示している |
| 7. 自分の言葉 | 説明文を自分の言葉で書いている（資料や AI の文をそのまま写していない） |

**提出前チェック**：(2) と (3) の ＿＿ をすべて埋めた／PDF は 5〜7 枚で，各スライドに図がある／授業スライド以外の出典が 2 つ以上ある